In [ ]:
# import subprocess 
# import os
# import time
# import pyautogui 
# import webbrowser

# Taking input

In [42]:
input_querry=input("enter your querry: ")

# Making prompt

In [ ]:
SYSTEM_INSTRUCTIONS = """You are the task-planning brain for a laptop automation tool.

Given a user's natural language request, break it down into an ordered sequence of actions.

Available action_type values and what their `params` should contain:
- open_application: params = {"app": "<application name, e.g. chrome, notepad, vscode>"}
- search: params = {"platform": "<google/youtube/etc>", "query": "<search text>"}
- create_folder: params = {"path": "<folder name or path>"}
- move_file: params = {"source": "<file/folder name>", "destination": "<target folder>"}
- run_command: params = {"command": "<shell command>"}   # reserved for future use
- open_website: params = {"url": "<full url>"}            # reserved for future use
- gui_action: params = {}                                  # reserved for future use

Rules:
- If the user wants to search the web, use action_type "search" — do not use open_website unless they give an explicit URL.
- Number actions in the order they should execute, starting from step_id = 1.
- If a required piece of information is missing or unclear (e.g. "open my project" with no project name),
  set needs_clarification = true on that action and write a specific clarification_question.
  Do not guess a value you are not reasonably confident about.
- goal should be a short, plain-language summary of what the user is trying to accomplish overall.
"""

def build_prompt(user_query: str) -> str:
    return f"{SYSTEM_INSTRUCTIONS}\n\nUser request: \"{user_query}\""

In [44]:
prompt = build_prompt(input_querry)

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

api_key = os.getenv("Gemini_api")

In [46]:
from google import genai
client = genai.Client(api_key=api_key)

In [ ]:
from enum import Enum
from typing import Any, Dict, List, Optional
from pydantic import BaseModel, Field, model_validator


class ActionType(str, Enum):
    OPEN_APPLICATION = "open_application"
    SEARCH = "search"
    CREATE_FOLDER = "create_folder"
    MOVE_FILE = "move_file"
    RUN_COMMAND = "run_command"      # placeholder for Phase 4
    OPEN_WEBSITE = "open_website"    # placeholder for Phase 5
    GUI_ACTION = "gui_action"        # placeholder for Phase 6


class Action(BaseModel):
    step_id: int
    action_type: ActionType
    params: Dict[str, Any] = Field(default_factory=dict)
    needs_clarification: bool = False
    clarification_question: Optional[str] = None

    @model_validator(mode="after")
    def check_clarification_consistency(self):
        if self.needs_clarification and not self.clarification_question:
            raise ValueError(
                "clarification_question is required when needs_clarification is True"
            )
        return self


class TaskPlan(BaseModel):
    goal: str
    actions: List[Action]

In [47]:
def response_maker(prompt: str) -> TaskPlan:
    interaction = client.interactions.create(
        model="gemini-3.5-flash",
        input=prompt,
        response_format={
            "type": "text",
            "mime_type": "application/json",
            "schema": TaskPlan.model_json_schema(),
        },
    )
    return TaskPlan.model_validate_json(interaction.output_text)

In [40]:
test_inputs = [
    # Should trigger needs_clarification
    "Search for the best C++ tutorials",
    "Look up the weather",
    "Search YouTube",
    "Open my project",
    "Find my resume",
    "Open the document",
    "Move my files to the new folder",
    "Search for flights",
    "Open it",
    "Create a folder for my notes",
    "Search for a recipe",
    "Open the app I used yesterday",
    "Find that PDF I downloaded",

    # Should NOT trigger clarification
    "Open Chrome and search Google for Python documentation",
    "Open Notepad and write Hello World",
    "Search YouTube for C++ tutorials",
    "Create a folder called AI Projects",
    "Open VS Code",
    "Search Google for machine learning courses",
    "Open Spotify and search for lofi playlists",

    # Edge cases
    "Open Chrome and search for Python documentation, also open my RAG project",
    "Open my project called RAG",
    "Clean up my downloads folder",
    "Search for it again",
    "Open the browser",
]

results = []

for i, query in enumerate(test_inputs, start=1):
    prompt = build_prompt(query)
    try:
        plan = response_maker(prompt)
        results.append({"input": query, "status": "ok", "plan": plan})
    except Exception as e:
        results.append({"input": query, "status": "error", "error": str(e)})

for r in results:
    print(f"\n--- Input: {r['input']} ---")
    if r["status"] == "error":
        print(f"  FAILED TO PARSE/VALIDATE: {r['error']}")
        continue

    plan = r["plan"]
    print(f"  goal: {plan.goal}")
    for action in plan.actions:
        flag = "⚠️ NEEDS CLARIFICATION" if action.needs_clarification else "✅"
        print(f"    step {action.step_id}: {action.action_type.value} | {flag}")
        print(f"      params: {action.params}")
        if action.needs_clarification:
            print(f"      question: {action.clarification_question}")


--- Input: Search for the best C++ tutorials ---
  goal: Search for the best C++ tutorials
    step 1: search | ✅
      params: {'platform': 'google', 'query': 'best C++ tutorials'}

--- Input: Look up the weather ---
  goal: Look up the current weather
    step 1: search | ✅
      params: {'platform': 'google', 'query': 'weather'}

--- Input: Search YouTube ---
  FAILED TO PARSE/VALIDATE: 1 validation error for TaskPlan
actions.0
  Value error, clarification_question is required when needs_clarification is True [type=value_error, input_value={'step_id': 1, 'action_ty...outube', 'query': None}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

--- Input: Open my project ---
  FAILED TO PARSE/VALIDATE: 2 validation errors for TaskPlan
actions.0
  Value error, clarification_question is required when needs_clarification is True [type=value_error, input_value={'step_id': 1, 'action_ty..., 'params': {'app': ''}}, input_type=dict]
    For fu